In [ ]:
!wget http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar
!tar -xf images.tar

--2026-05-26 23:12:47--  http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar
Resolving vision.stanford.edu (vision.stanford.edu)... 171.64.68.10
Connecting to vision.stanford.edu (vision.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 793579520 (757M) [application/x-tar]
Saving to: ‘images.tar.2’

images.tar.2        100%[===================>] 756.82M  12.5MB/s    in 79s     

2026-05-26 23:14:06 (9.62 MB/s) - ‘images.tar.2’ saved [793579520/793579520]



In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import ResNeXt50_32X4D_Weights
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CustomDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

full_dataset = ImageFolder(root='Images')
total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = CustomDataset(train_subset, transform=train_transform)
val_dataset = CustomDataset(val_subset, transform=test_transform)
test_dataset = CustomDataset(test_subset, transform=test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Размеры выборок - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

class ResNeXtBlock(nn.Module):
    expansion = 4

    def __init__(self, in_channels, planes, stride, cardinality, base_width):
        super().__init__()

        width = int(planes * (base_width / 64.0)) * cardinality

        self.conv1 = nn.Conv2d(in_channels, width, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)

        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=stride,
                               padding=1, groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(width)

        self.conv3 = nn.Conv2d(width, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)

        self.relu = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_channels != planes * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion)
            )

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

class ResNeXt(nn.Module):
    def __init__(self, block, num_blocks, num_classes, cardinality, base_width):
        super().__init__()
        self.in_channels = 64
        self.cardinality = cardinality
        self.base_width = base_width

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, num_blocks[0], planes=64, stride=1)
        self.layer2 = self._make_layer(block, num_blocks[1], planes=128, stride=2)
        self.layer3 = self._make_layer(block, num_blocks[2], planes=256, stride=2)
        self.layer4 = self._make_layer(block, num_blocks[3], planes=512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        self._initialize_weights()

    def _make_layer(self, block, num_blocks, planes, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, planes, s, self.cardinality, self.base_width))
            self.in_channels = planes * block.expansion
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

def get_model(pretrained=True):
    model = ResNeXt(ResNeXtBlock, num_blocks=[3, 4, 6, 3], num_classes=120,
                    cardinality=32, base_width=4)

    if pretrained:
        print("Загрузка предобученных весов...")
        official_state_dict = models.resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.DEFAULT).state_dict()

        del official_state_dict['fc.weight']
        del official_state_dict['fc.bias']

        model.load_state_dict(official_state_dict, strict=False)
    else:
        print("Инициализация модели")

    return model.to(device)

def train_and_evaluate(model, optimizer, criterion, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='macro', zero_division=0)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {total_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | "
              f"Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")
    return model

def test_model(model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)
    return precision, recall, f1

criterion = nn.CrossEntropyLoss()
epochs = 5

print("\n--- Эксперимент 1: Pretrained ResNeXt + Adam ---")
model_exp1 = get_model(pretrained=True)
optimizer_exp1 = torch.optim.Adam(model_exp1.parameters(), lr=1e-4)
train_and_evaluate(model_exp1, optimizer_exp1, criterion, epochs)
metrics_exp1 = test_model(model_exp1)

print("\n--- Эксперимент 2: Pretrained ResNeXt + RAdam ---")
model_exp2 = get_model(pretrained=True)
optimizer_exp2 = torch.optim.RAdam(model_exp2.parameters(), lr=1e-4)
train_and_evaluate(model_exp2, optimizer_exp2, criterion, epochs)
metrics_exp2 = test_model(model_exp2)

print("\n--- Эксперимент 3: Своя архитектура с нуля (Scratch) + Adam ---")
model_exp3 = get_model(pretrained=False)
optimizer_exp3 = torch.optim.Adam(model_exp3.parameters(), lr=1e-4)
train_and_evaluate(model_exp3, optimizer_exp3, criterion, epochs)
metrics_exp3 = test_model(model_exp3)

print("\n==============================================")
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ (TEST SET):")
print(f"1. Pretrained + Adam : Prec: {metrics_exp1[0]:.4f}, Rec: {metrics_exp1[1]:.4f}, F1: {metrics_exp1[2]:.4f}")
print(f"2. Pretrained + RAdam: Prec: {metrics_exp2[0]:.4f}, Rec: {metrics_exp2[1]:.4f}, F1: {metrics_exp2[2]:.4f}")
print(f"3. Scratch + Adam    : Prec: {metrics_exp3[0]:.4f}, Rec: {metrics_exp3[1]:.4f}, F1: {metrics_exp3[2]:.4f}")
print("==============================================")

labels = ['Adam Fine-tune', 'RAdam Fine-tune', 'Scratch Adam']
precision_scores = [metrics_exp1[0], metrics_exp2[0], metrics_exp3[0]]
recall_scores = [metrics_exp1[1], metrics_exp2[1], metrics_exp3[1]]
f1_scores = [metrics_exp1[2], metrics_exp2[2], metrics_exp3[2]]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7), dpi=100)

rects1 = ax.bar(x - width, precision_scores, width, label='Precision', color='#3498db')
rects2 = ax.bar(x, recall_scores, width, label='Recall', color='#2ecc71')
rects3 = ax.bar(x + width, f1_scores, width, label='F1-score', color='#e74c3c')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Сравнение моделей на тестовой выборке Stanford Dogs', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.0) # Шкала от 0 до 1
ax.legend(loc='lower right', fontsize=11)

ax.grid(axis='y', linestyle='-', alpha=0.3)

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

fig.tight_layout()

plt.savefig('comparison_results.png')
plt.show()

Используемое устройство: cuda
Размеры выборок - Train: 14405, Val: 3087, Test: 3088

--- Эксперимент 1: Pretrained ResNeXt + Adam ---
Загрузка предобученных весов...
Веса успешно перенесены в кастомную архитектуру!
Epoch 1/5 | Train Loss: 2.0139 | Val Loss: 0.5746 | Prec: 0.8536 | Rec: 0.8486 | F1: 0.8443
Epoch 2/5 | Train Loss: 0.5893 | Val Loss: 0.5047 | Prec: 0.8557 | Rec: 0.8485 | F1: 0.8463
Epoch 3/5 | Train Loss: 0.3767 | Val Loss: 0.4947 | Prec: 0.8573 | Rec: 0.8516 | F1: 0.8486
Epoch 4/5 | Train Loss: 0.2809 | Val Loss: 0.5484 | Prec: 0.8474 | Rec: 0.8407 | F1: 0.8380
Epoch 5/5 | Train Loss: 0.2109 | Val Loss: 0.5211 | Prec: 0.8495 | Rec: 0.8429 | F1: 0.8406

--- Эксперимент 2: Pretrained ResNeXt + RAdam ---
Загрузка предобученных весов...
Веса успешно перенесены в кастомную архитектуру!
Epoch 1/5 | Train Loss: 3.8874 | Val Loss: 1.5478 | Prec: 0.7806 | Rec: 0.7578 | F1: 0.7458
Epoch 2/5 | Train Loss: 1.0494 | Val Loss: 0.5704 | Prec: 0.8498 | Rec: 0.8459 | F1: 0.8393
Epoch 3/5